# Set 8 — Local CNN Training, Model Passports & Feature Extraction

This notebook covers the **training and export** requirements for **Lab / Set 8** locally (no Google Colab required), producing all artifacts needed by the **ForensicsApp** tools in **Set 8.2** and **Set 8.3**.

### What this notebook builds and saves
1. **Set 8.1a — Small Classification CNN** trained from scratch on CIFAR-10:
   - `models/set8/small_cnn_cifar10.keras` (trained model + weights)
   - `models/set8/small_cnn_cifar10_passport.json` (model passport: dataset, sample sizes, class names, preprocessing instructions, training history, test metrics)
2. **Set 8.1b — Transfer Learning (`InceptionV3`)** adapted to CIFAR-10:
   - `models/set8/inceptionv3_cifar10.keras`
   - `models/set8/inceptionv3_cifar10_passport.json`
3. **Set 8.3 (Prep) — Test-Set Feature Vectors** for 2D projection (`t-SNE` and `UMAP`) inside ForensicsApp:
   - `models/set8/small_cnn_test_features.npz`
   - `models/set8/inceptionv3_test_features.npz`
4. **Sample CIFAR-10 Test Images** saved to `models/set8/sample_images/` so you can immediately test **Set 8.2** inside ForensicsApp.

### Prerequisites
Make sure you have TensorFlow and UMAP installed in your environment:
```bash
pip install tensorflow umap-learn
```

## 1. Setup, Paths & Configuration

We detect the repository root automatically (whether you launch Jupyter from the repo root or from `notebooks/`), set random seeds for reproducibility, and create `models/set8/` for all exported artifacts.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import confusion_matrix
import tensorflow as tf
from tensorflow import keras

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Locate project root whether notebook is run from repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
OUTPUT_DIR = PROJECT_ROOT / "models" / "set8"
SAMPLE_IMG_DIR = OUTPUT_DIR / "sample_images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_IMG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

print(f"TensorFlow version : {tf.__version__}")
print(f"Available GPUs     : {tf.config.list_physical_devices('GPU')}")
print(f"Output directory   : {OUTPUT_DIR}")

## 2. Load & Explore CIFAR-10

CIFAR-10 contains **60,000** color (`32x32x3`) images across 10 balanced classes:
- **50,000** training images
- **10,000** test images

Below we:
1. Load the raw `uint8` arrays.
2. Visualize 5 sample images per class.
3. Save 10 sample test images (one per class) as PNGs in `models/set8/sample_images/` so you can load them into **ForensicsApp** to verify **Set 8.2**.

In [ ]:
from tensorflow.keras.datasets import cifar10

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = cifar10.load_data()
y_train_idx = y_train_raw.squeeze()
y_test_idx = y_test_raw.squeeze()

print("x_train:", x_train_raw.shape, x_train_raw.dtype)
print("x_test :", x_test_raw.shape, x_test_raw.dtype)

# Visualize 5 samples per class
fig, axes = plt.subplots(10, 5, figsize=(7, 13))
for cls_id, cls_name in enumerate(CLASS_NAMES):
    idxs = np.where(y_train_idx == cls_id)[0][:5]
    for col, idx in enumerate(idxs):
        axes[cls_id, col].imshow(x_train_raw[idx])
        axes[cls_id, col].axis("off")
        if col == 0:
            axes[cls_id, col].set_title(cls_name, loc="left", fontsize=9)
plt.tight_layout()
plt.show()

# Save 1 sample test image per class for quick testing in ForensicsApp (Set 8.2)
for cls_id, cls_name in enumerate(CLASS_NAMES):
    idx = int(np.where(y_test_idx == cls_id)[0][0])
    out_path = SAMPLE_IMG_DIR / f"test_{cls_id}_{cls_name}.png"
    Image.fromarray(x_test_raw[idx]).save(out_path)
print(f"Saved 10 sample test images to {SAMPLE_IMG_DIR}")

### Helper Functions: Plotting & Model Passport Export

A **Model Passport** (Set 8.1 & Set 8.2) is a structured JSON file stored alongside the `.keras` model. It records everything a user—or the ForensicsApp—needs to understand and correctly run the model:
- **Dataset & sample sizes** (train / validation / test counts)
- **Class names** (index-to-label mapping)
- **Preprocessing instructions** (expected input shape, color order, pixel scaling rule)
- **Training history** (per-epoch `loss`, `accuracy`, `val_loss`, `val_accuracy`)

In [ ]:
def plot_history(history, title: str):
    hist = history.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(hist["loss"], label="train")
    if "val_loss" in hist:
        ax1.plot(hist["val_loss"], label="val")
    ax1.set_title(f"{title} — Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Categorical Crossentropy")
    ax1.legend()

    ax2.plot(hist["accuracy"], label="train")
    if "val_accuracy" in hist:
        ax2.plot(hist["val_accuracy"], label="val")
    ax2.set_title(f"{title} — Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names, title: str):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap=plt.cm.Blues)
    thresh = cm.max() / 2.0
    for i, j in np.ndindex(cm.shape):
        ax.text(
            j, i, int(cm[i, j]),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=8,
        )
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def save_model_and_passport(
    model,
    stem: str,
    *, 
    model_name: str,
    architecture: str,
    dataset: str,
    sample_size: dict,
    class_names: list,
    preprocessing: dict,
    history_dict: dict,
    test_metrics: dict,
):
    model_path = OUTPUT_DIR / f"{stem}.keras"
    passport_path = OUTPUT_DIR / f"{stem}_passport.json"

    model.save(model_path)

    # Convert numpy floats to plain Python floats for JSON serialization
    clean_history = {
        k: [float(v) for v in vals]
        for k, vals in history_dict.items()
    }
    passport = {
        "model_name": model_name,
        "architecture": architecture,
        "model_file": model_path.name,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "framework": f"tensorflow {tf.__version__} / keras {keras.__version__}",
        "dataset": dataset,
        "sample_size": sample_size,
        "class_names": list(class_names),
        "preprocessing": preprocessing,
        "training_history": clean_history,
        "test_metrics": {k: float(v) for k, v in test_metrics.items()},
    }
    passport_path.write_text(json.dumps(passport, indent=2), encoding="utf-8")
    print(f"Saved model    -> {model_path}")
    print(f"Saved passport -> {passport_path}")
    return model_path, passport_path

---
## 3. Set 8.1a — Train & Save a Small Classification CNN

Here we train a compact VGG-style CNN directly on `32x32x3` CIFAR-10 images normalized to `[0, 1]`.
- We use the **Adam** optimizer (`lr=1e-3`) instead of plain SGD so it reaches strong accuracy (~70–75%) in just **15 epochs**, which runs in ~1–3 minutes on a multi-core CPU.
- The penultimate dense layer is named `feature_dense` so we can easily extract 128-D feature vectors from it later in **Section 5**.

In [ ]:
keras.backend.clear_session()

x_train_cnn = x_train_raw.astype("float32") / 255.0
x_test_cnn = x_test_raw.astype("float32") / 255.0
y_train_oh = keras.utils.to_categorical(y_train_idx, num_classes=10)
y_test_oh = keras.utils.to_categorical(y_test_idx, num_classes=10)

small_cnn = keras.Sequential(
    [
        keras.layers.Input(shape=(32, 32, 3)),
        keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same", kernel_initializer="he_uniform"),
        keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same", kernel_initializer="he_uniform"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.2),
        keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same", kernel_initializer="he_uniform"),
        keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same", kernel_initializer="he_uniform"),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Dropout(0.25),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu", kernel_initializer="he_uniform", name="feature_dense"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(10, activation="softmax", name="predictions"),
    ],
    name="small_cnn_cifar10",
)

small_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
small_cnn.summary()

In [ ]:
SMALL_CNN_EPOCHS = 15
SMALL_CNN_BATCH_SIZE = 64
VAL_SPLIT = 0.1

history_small = small_cnn.fit(
    x_train_cnn,
    y_train_oh,
    epochs=SMALL_CNN_EPOCHS,
    batch_size=SMALL_CNN_BATCH_SIZE,
    validation_split=VAL_SPLIT,
    verbose=1,
)

plot_history(history_small, "Small CNN (CIFAR-10)")

In [ ]:
# Evaluate on the full CIFAR-10 test set
test_loss, test_acc = small_cnn.evaluate(x_test_cnn, y_test_oh, verbose=0)
print(f"Small CNN Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

y_pred_small = np.argmax(small_cnn.predict(x_test_cnn, verbose=0), axis=1)
plot_confusion_matrix(y_test_idx, y_pred_small, CLASS_NAMES, "Small CNN — Confusion Matrix")

# Save model + passport
n_val = int(len(x_train_cnn) * VAL_SPLIT)
n_train = len(x_train_cnn) - n_val

save_model_and_passport(
    small_cnn,
    stem="small_cnn_cifar10",
    model_name="Small VGG-style CNN (CIFAR-10)",
    architecture="2x[Conv2D(32)] -> MaxPool -> 2x[Conv2D(64)] -> MaxPool -> Dense(128) -> Dense(10)",
    dataset="CIFAR-10",
    sample_size={
        "train": n_train,
        "validation": n_val,
        "test": int(len(x_test_cnn)),
        "total": int(len(x_train_cnn) + len(x_test_cnn)),
    },
    class_names=CLASS_NAMES,
    preprocessing={
        "input_height": 32,
        "input_width": 32,
        "channels": 3,
        "color_mode": "RGB",
        "normalization": "rescale_0_1",
        "description": "Convert to RGB, resize to 32x32, cast to float32, divide pixel values by 255.0 (range [0, 1]).",
    },
    history_dict=history_small.history,
    test_metrics={"test_loss": test_loss, "test_accuracy": test_acc},
)

---
## 4. Set 8.1b — Transfer Learning with `InceptionV3` & Save

Instead of training a large network from scratch, **transfer learning** reuses convolutional feature extractors already trained on ImageNet (`1.2M` images, `1,000` classes):
1. Load `InceptionV3(include_top=False, weights='imagenet')` and **freeze** its layers.
2. Attach a small classification head (`GlobalAveragePooling2D` → `Dense(256, relu)` → `Dropout(0.3)` → `Dense(10, softmax)`).
3. Resize a **stratified subset** of CIFAR-10 to `128x128x3` (InceptionV3 requires inputs `>= 75x75`) and apply `inception_v3.preprocess_input` (scales `[0, 255]` → `[-1, 1]`).
4. Train the classification head for a few epochs so it runs quickly on CPU (you can increase `N_TRAIN_PER_CLASS` if you have extra time or a GPU).

In [ ]:
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess

keras.backend.clear_session()

INCEPTION_IMG_SIZE = 128
N_TRAIN_PER_CLASS = 200  # 2,000 training images total (fast on CPU; increase for higher accuracy)
N_TEST_PER_CLASS = 50    # 500 test images total


def stratified_subset(x_raw, y_idx, n_per_class: int, img_size: int):
    xs, ys = [], []
    for cls_id in range(10):
        idxs = np.where(y_idx == cls_id)[0][:n_per_class]
        xs.append(x_raw[idxs])
        ys.append(y_idx[idxs])
    x_sub = np.concatenate(xs, axis=0)
    y_sub = np.concatenate(ys, axis=0)
    # Shuffle deterministically
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(x_sub))
    x_sub, y_sub = x_sub[perm], y_sub[perm]
    # Fast vectorized resize with TensorFlow + InceptionV3 [-1, 1] preprocessing
    x_resized = tf.image.resize(x_sub, (img_size, img_size)).numpy().astype("float32")
    x_pre = inception_preprocess(x_resized)
    return x_pre, y_sub


x_train_inc, y_train_inc_idx = stratified_subset(x_train_raw, y_train_idx, N_TRAIN_PER_CLASS, INCEPTION_IMG_SIZE)
x_test_inc, y_test_inc_idx = stratified_subset(x_test_raw, y_test_idx, N_TEST_PER_CLASS, INCEPTION_IMG_SIZE)
y_train_inc_oh = keras.utils.to_categorical(y_train_inc_idx, num_classes=10)
y_test_inc_oh = keras.utils.to_categorical(y_test_inc_idx, num_classes=10)

print("InceptionV3 train subset:", x_train_inc.shape, "range:", x_train_inc.min(), "..", x_train_inc.max())
print("InceptionV3 test subset :", x_test_inc.shape)

base_model = InceptionV3(
    include_top=False,
    weights="imagenet",
    input_shape=(INCEPTION_IMG_SIZE, INCEPTION_IMG_SIZE, 3),
)
base_model.trainable = False

inputs = keras.Input(shape=(INCEPTION_IMG_SIZE, INCEPTION_IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = keras.layers.Dense(256, activation="relu", name="custom_dense")(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(10, activation="softmax", name="predictions")(x)

inception_clf = keras.Model(inputs=inputs, outputs=outputs, name="inceptionv3_cifar10")
inception_clf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
inception_clf.summary()

In [ ]:
INCEPTION_EPOCHS = 8
INCEPTION_BATCH_SIZE = 32
INC_VAL_SPLIT = 0.1

history_inc = inception_clf.fit(
    x_train_inc,
    y_train_inc_oh,
    epochs=INCEPTION_EPOCHS,
    batch_size=INCEPTION_BATCH_SIZE,
    validation_split=INC_VAL_SPLIT,
    verbose=1,
)

plot_history(history_inc, "InceptionV3 Transfer Learning (CIFAR-10)")

In [ ]:
inc_test_loss, inc_test_acc = inception_clf.evaluate(x_test_inc, y_test_inc_oh, verbose=0)
print(f"InceptionV3 Test Loss: {inc_test_loss:.4f} | Test Accuracy: {inc_test_acc:.4f}")

y_pred_inc = np.argmax(inception_clf.predict(x_test_inc, verbose=0), axis=1)
plot_confusion_matrix(y_test_inc_idx, y_pred_inc, CLASS_NAMES, "InceptionV3 — Confusion Matrix")

n_inc_val = int(len(x_train_inc) * INC_VAL_SPLIT)
n_inc_train = len(x_train_inc) - n_inc_val

save_model_and_passport(
    inception_clf,
    stem="inceptionv3_cifar10",
    model_name="InceptionV3 Transfer Learning (CIFAR-10)",
    architecture="InceptionV3(imagenet, frozen) -> GlobalAveragePooling2D -> Dense(256) -> Dropout(0.3) -> Dense(10)",
    dataset="CIFAR-10 (stratified subset)",
    sample_size={
        "train": n_inc_train,
        "validation": n_inc_val,
        "test": int(len(x_test_inc)),
        "total": int(len(x_train_inc) + len(x_test_inc)),
    },
    class_names=CLASS_NAMES,
    preprocessing={
        "input_height": INCEPTION_IMG_SIZE,
        "input_width": INCEPTION_IMG_SIZE,
        "channels": 3,
        "color_mode": "RGB",
        "normalization": "inception_v3_minus1_1",
        "description": f"Convert to RGB, resize to {INCEPTION_IMG_SIZE}x{INCEPTION_IMG_SIZE}, cast to float32, scale to [-1, 1] via (x / 127.5) - 1.0.",
    },
    history_dict=history_inc.history,
    test_metrics={"test_loss": inc_test_loss, "test_accuracy": inc_test_acc},
)

---
## 5. Set 8.3 (Prep) — Extract & Save Test-Set Feature Vectors

For **Set 8.3 (Feature Projection: t-SNE & UMAP)**, we extract high-dimensional feature embeddings from the **CIFAR-10 test set** and save them to compressed `.npz` files so ForensicsApp can load them instantly without re-running the CNN.

We save two feature sets:
1. `models/set8/small_cnn_test_features.npz` — 128-D embeddings from `small_cnn_cifar10`'s `feature_dense` layer (`1,000` stratified test images: `100` per class).
2. `models/set8/inceptionv3_test_features.npz` — 2048-D embeddings from `inceptionv3_cifar10`'s `global_avg_pool` layer (`500` stratified test images: `50` per class).

In [ ]:
# 1) Extract 128-D test features from the trained Small CNN
N_FEAT_PER_CLASS = 100  # 1,000 stratified test images
small_cnn_loaded = keras.models.load_model(OUTPUT_DIR / "small_cnn_cifar10.keras")
small_feat_extractor = keras.Model(
    inputs=small_cnn_loaded.inputs,
    outputs=small_cnn_loaded.get_layer("feature_dense").output,
)

feat_xs, feat_ys = [], []
for cls_id in range(10):
    idxs = np.where(y_test_idx == cls_id)[0][:N_FEAT_PER_CLASS]
    feat_xs.append(x_test_cnn[idxs])
    feat_ys.append(y_test_idx[idxs])
x_feat_small = np.concatenate(feat_xs, axis=0)
y_feat_small = np.concatenate(feat_ys, axis=0)

small_features = small_feat_extractor.predict(x_feat_small, batch_size=64, verbose=0)
small_feat_path = OUTPUT_DIR / "small_cnn_test_features.npz"
np.savez_compressed(
    small_feat_path,
    features=small_features.astype("float32"),
    labels=y_feat_small.astype("int32"),
    class_names=np.array(CLASS_NAMES),
    model_name=np.array("Small VGG-style CNN (CIFAR-10)"),
    layer_name=np.array("feature_dense"),
    dataset=np.array("CIFAR-10 test set"),
)
print(f"Saved Small CNN features {small_features.shape} -> {small_feat_path}")

# 2) Extract 2048-D test features from InceptionV3's global_avg_pool layer
inc_feat_extractor = keras.Model(
    inputs=inception_clf.inputs,
    outputs=inception_clf.get_layer("global_avg_pool").output,
)
inc_features = inc_feat_extractor.predict(x_test_inc, batch_size=32, verbose=0)
inc_feat_path = OUTPUT_DIR / "inceptionv3_test_features.npz"
np.savez_compressed(
    inc_feat_path,
    features=inc_features.astype("float32"),
    labels=y_test_inc_idx.astype("int32"),
    class_names=np.array(CLASS_NAMES),
    model_name=np.array("InceptionV3 Transfer Learning (CIFAR-10)"),
    layer_name=np.array("global_avg_pool"),
    dataset=np.array("CIFAR-10 test set"),
)
print(f"Saved InceptionV3 features {inc_features.shape} -> {inc_feat_path}")

### Quick Verification: Preview t-SNE & UMAP Projections

Run this cell to verify that your saved test feature vectors separate nicely in 2D before loading them into ForensicsApp (Set 8.3).

In [ ]:
from sklearn.manifold import TSNE
import umap

data = np.load(OUTPUT_DIR / "small_cnn_test_features.npz", allow_pickle=False)
feats = data["features"]
labels = data["labels"]
names = [str(c) for c in data["class_names"]]

tsne_2d = TSNE(n_components=2, perplexity=30, random_state=SEED).fit_transform(feats)
umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(feats)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for cls_id, cls_name in enumerate(names):
    mask = labels == cls_id
    ax1.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1], s=10, alpha=0.75, label=cls_name)
    ax2.scatter(umap_2d[mask, 0], umap_2d[mask, 1], s=10, alpha=0.75, label=cls_name)

ax1.set_title("t-SNE (Small CNN test features)")
ax1.set_xlabel("t-SNE 1")
ax1.set_ylabel("t-SNE 2")

ax2.set_title("UMAP (Small CNN test features)")
ax2.set_xlabel("UMAP 1")
ax2.set_ylabel("UMAP 2")
ax2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, markerscale=2)

plt.tight_layout()
plt.show()